In [1]:
import os
import json
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual para ver todas las columnas
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

print("✅ Entorno listo y librerías importadas.")
print("Versión de Pandas:", pd.__version__)

✅ Entorno listo y librerías importadas.
Versión de Pandas: 3.1.1


In [2]:
# 1. Crear las carpetas locales si no existen
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

raw_dir = "../data/raw"
dataset_api_url = "https://huggingface.co/api/datasets/takara-ai/poker_hands"

print("🔍 Consultando la lista de archivos Parquet en Hugging Face...")
req = urllib.request.Request(dataset_api_url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as response:
    repo_info = json.loads(response.read().decode('utf-8'))

# 2. Filtrar únicamente los archivos parquet de datos
parquet_files = [
    f["rfilename"] for f in repo_info.get("siblings", [])
    if f["rfilename"].endswith(".parquet") and "data/" in f["rfilename"]
]

print(f"📦 Total de archivos Parquet detectados en el repositorio: {len(parquet_files)}")
target_file = parquet_files[0]
print(f"🎯 Archivo seleccionado para nuestra muestra: {target_file}")

# 3. Descargar el archivo seleccionado
local_filename = os.path.join(raw_dir, os.path.basename(target_file))
download_url = f"https://huggingface.co/datasets/takara-ai/poker_hands/resolve/main/{target_file}"

if not os.path.exists(local_filename):
    print(f"⬇️ Descargando {target_file} (esto puede tomar 1 o 2 minutos)...")
    
    def progress_hook(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            percent = min(100.0, downloaded * 100.0 / total_size)
            mb_down = downloaded / (1024 * 1024)
            mb_total = total_size / (1024 * 1024)
            print(f"\r   Progreso: {percent:.1f}% ({mb_down:.1f} MB de {mb_total:.1f} MB)", end="")
        else:
            print(f"\r   Descargados: {downloaded / (1024 * 1024):.1f} MB", end="")
            
    urllib.request.urlretrieve(download_url, local_filename, progress_hook)
    print("\n✅ Descarga completada con éxito.")
else:
    print(f"⚡ El archivo ya existe localmente en: {local_filename}")

file_size_mb = os.path.getsize(local_filename) / (1024 * 1024)
print(f"📁 Tamaño del archivo guardado: {file_size_mb:.2f} MB")

🔍 Consultando la lista de archivos Parquet en Hugging Face...
📦 Total de archivos Parquet detectados en el repositorio: 37
🎯 Archivo seleccionado para nuestra muestra: data/train-00000-of-00037.parquet
⬇️ Descargando data/train-00000-of-00037.parquet (esto puede tomar 1 o 2 minutos)...
   Progreso: 100.0% (27.9 MB de 27.9 MB)
✅ Descarga completada con éxito.
📁 Tamaño del archivo guardado: 27.94 MB


In [3]:
print(f"📖 Abriendo el archivo local con Pandas: {local_filename}")

# Cargar el archivo en un DataFrame de Pandas
df = pd.read_parquet(local_filename)

print("\n--- RESUMEN BÁSICO DE LA MUESTRA ---")
print(f"Total de manos registradas en este archivo: {len(df):,}")
print(f"Total de columnas: {len(df.columns)}")
print(f"Memoria utilizada en RAM: {df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")

print("\n--- COLUMNAS ENCONTRADAS Y SUS TIPOS DE DATO ---")
for i, col in enumerate(df.columns, 1):
    print(f" {i}. {col:<25} -> tipo: {df[col].dtype}")

📖 Abriendo el archivo local con Pandas: ../data/raw\train-00000-of-00037.parquet

--- RESUMEN BÁSICO DE LA MUESTRA ---
Total de manos registradas en este archivo: 450,337
Total de columnas: 35
Memoria utilizada en RAM: 564.03 MB

--- COLUMNAS ENCONTRADAS Y SUS TIPOS DE DATO ---
 1. variant                   -> tipo: str
 2. venue                     -> tipo: str
 3. table                     -> tipo: str
 4. time                      -> tipo: str
 5. time_zone_abbreviation    -> tipo: str
 6. currency_symbol           -> tipo: str
 7. currency                  -> tipo: str
 8. author                    -> tipo: str
 9. event                     -> tipo: str
 10. city                      -> tipo: str
 11. region                    -> tipo: str
 12. country                   -> tipo: str
 13. address                   -> tipo: str
 14. postal_code               -> tipo: str
 15. source_file               -> tipo: str
 16. source_type               -> tipo: str
 17. hand                 

In [4]:
# 1. Ver la lista completa de las 35 columnas sin que VS Code las recorte
print("📋 LAS 35 COLUMNAS DEL DATASET:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:02d}. {col}")

# 2. Auditoría: ¿Qué variantes de póker contiene este archivo?
print("\n🃏 DISTRIBUCIÓN POR VARIANTE (variant):")
print(df['variant'].value_counts())

# 3. Auditoría: ¿De qué salas de póker provienen?
print("\n🏛️ DISTRIBUCIÓN POR SALA (venue):")
print(df['venue'].value_counts().head(10))

📋 LAS 35 COLUMNAS DEL DATASET:
  01. variant
  02. venue
  03. table
  04. time
  05. time_zone_abbreviation
  06. currency_symbol
  07. currency
  08. author
  09. event
  10. city
  11. region
  12. country
  13. address
  14. postal_code
  15. source_file
  16. source_type
  17. hand
  18. hand_number_in_file
  19. day
  20. month
  21. year
  22. seat_count
  23. min_bet
  24. small_bet
  25. big_bet
  26. bring_in
  27. ante_trimming_status
  28. actions
  29. players
  30. antes
  31. blinds_or_straddles
  32. starting_stacks
  33. finishing_stacks
  34. winnings
  35. seats

🃏 DISTRIBUCIÓN POR VARIANTE (variant):
variant
NT       450261
FO/8         14
F7S          13
FR           10
PO            8
F2L3D         8
N2L1D         7
FT            7
F7S/8         7
FB            1
NS            1
Name: count, dtype: int64

🏛️ DISTRIBUCIÓN POR SALA (venue):
venue
Full Tilt Poker    373419
Absolute Poker      66830
Name: count, dtype: int64


In [5]:
# 1. Porcentaje exacto de variantes de póker
print("🃏 VARIANTES DE PÓKER EN LA MUESTRA:")
for var, count in df['variant'].value_counts().items():
    porcentaje = (count / len(df)) * 100
    print(f"   • {var}: {count:,} manos ({porcentaje:.2f}%)")

# 2. Las columnas 25 a 35 que faltaban
print("\n📋 COLUMNAS CLAVE DE JUEGO (de la 25 a la 35):")
for i, col in enumerate(df.columns[24:], 25):
    print(f"   {i:02d}. {col}")

# 3. Ver una mano real (Fila 0) en las columnas más importantes
print("\n🔍 CÓMO SE VE UNA MANO REAL (Fila 0):")
cols_interes = ['variant', 'seat_count', 'starting_stacks', 'blinds', 'board', 'actions']

for col in cols_interes:
    if col in df.columns:
        print(f"\n👉 [{col}]:")
        print(f"   {df.iloc[0][col]}")

🃏 VARIANTES DE PÓKER EN LA MUESTRA:
   • NT: 450,261 manos (99.98%)
   • FO/8: 14 manos (0.00%)
   • F7S: 13 manos (0.00%)
   • FR: 10 manos (0.00%)
   • PO: 8 manos (0.00%)
   • F2L3D: 8 manos (0.00%)
   • N2L1D: 7 manos (0.00%)
   • FT: 7 manos (0.00%)
   • F7S/8: 7 manos (0.00%)
   • FB: 1 manos (0.00%)
   • NS: 1 manos (0.00%)

📋 COLUMNAS CLAVE DE JUEGO (de la 25 a la 35):
   25. big_bet
   26. bring_in
   27. ante_trimming_status
   28. actions
   29. players
   30. antes
   31. blinds_or_straddles
   32. starting_stacks
   33. finishing_stacks
   34. winnings
   35. seats

🔍 CÓMO SE VE UNA MANO REAL (Fila 0):

👉 [variant]:
   FB

👉 [seat_count]:
   nan

👉 [starting_stacks]:
   [200. 200. 200. 200.]

👉 [actions]:
   ['[']


In [6]:
# Tomamos la primera mano del dataset
mano = df.iloc[0]

print(f"🃏 VARIANTE: {mano['variant']}")
print(f"👥 JUGADORES EN LA MESA: {mano['players']}")
print(f"💰 FICHAS INICIALES (STACKS): {mano['starting_stacks']}")
print(f"🔴 CIEGAS (BLINDS): {mano['blinds_or_straddles']}")

# Desglosar la secuencia cronológica de acciones
acciones = mano['actions']
print(f"\n⚡ TOTAL DE PASOS/ACCIONES EN ESTA MANO: {len(acciones)}")
print("--- HISTORIA COMPLETA DE LA MANO ---")
for i, paso in enumerate(acciones, 1):
    print(f"  Paso {i:02d}: {paso}")

🃏 VARIANTE: FB
👥 JUGADORES EN LA MESA: ['Bob' 'Carol' 'Ted' 'Alice']
💰 FICHAS INICIALES (STACKS): [200. 200. 200. 200.]
🔴 CIEGAS (BLINDS): [1. 2. 0. 0.]

⚡ TOTAL DE PASOS/ACCIONES EN ESTA MANO: 1
--- HISTORIA COMPLETA DE LA MANO ---
  Paso 01: [


In [7]:
# 1. Filtro obligatorio: Solo No-Limit Texas Hold'em
df_nt = df[df['variant'] == 'NT'].copy()
print(f"✅ Manos de No-Limit Texas Hold'em listas: {len(df_nt):,}")

# 2. Tomamos la primera mano real de Texas Hold'em
mano_nt = df_nt.iloc[0]

print(f"\n🃏 VARIANTE: {mano_nt['variant']}")
print(f"🏛️ SALA: {mano_nt['venue']}")
print(f"👥 JUGADORES: {mano_nt['players']}")
print(f"💰 FICHAS INICIALES (STACKS): {mano_nt['starting_stacks']}")
print(f"🔴 CIEGAS (BLINDS): {mano_nt['blinds_or_straddles']}")

# 3. Inspeccionar las acciones de esta partida
acciones_nt = mano_nt['actions']
print(f"\nTipo de dato de 'actions': {type(acciones_nt)}")

print("\n📜 SECUENCIA DE ACCIONES DE LA PARTIDA:")
if hasattr(acciones_nt, '__iter__') and not isinstance(acciones_nt, str):
    print(f"Total de pasos registrados: {len(acciones_nt)}")
    for i, paso in enumerate(acciones_nt, 1):
        print(f"  Paso {i:02d}: {paso}")
else:
    print(acciones_nt)

✅ Manos de No-Limit Texas Hold'em listas: 450,261

🃏 VARIANTE: NT
🏛️ SALA: nan
👥 JUGADORES: ['Phil Ivey' 'Patrik Antonius' 'Tom Dwan']
💰 FICHAS INICIALES (STACKS): [1125600. 2000000.  553500.]
🔴 CIEGAS (BLINDS): [1000. 2000.    0.]

Tipo de dato de 'actions': <class 'numpy.ndarray'>

📜 SECUENCIA DE ACCIONES DE LA PARTIDA:
Total de pasos registrados: 1
  Paso 01: [


In [8]:
# Inspeccionar a fondo qué contiene el bloque de acciones
primer_elemento = acciones_nt[0]

print("Tipo de dato del primer elemento:", type(primer_elemento))
print("Longitud (caracteres o elementos):", len(primer_elemento))

print("\n--- PRIMEROS 600 CARACTERES DEL BLOQUE ---")
print(str(primer_elemento)[:600])

print("\n--- ÚLTIMOS 300 CARACTERES DEL BLOQUE ---")
print(str(primer_elemento)[-300:])

Tipo de dato del primer elemento: <class 'str'>
Longitud (caracteres o elementos): 1

--- PRIMEROS 600 CARACTERES DEL BLOQUE ---
[

--- ÚLTIMOS 300 CARACTERES DEL BLOQUE ---
[


In [9]:
# 1. Auditar la longitud del campo 'actions' en las primeras 10 manos
print("🔍 LONGITUD DE 'actions' EN LAS PRIMERAS 10 FILAS:")
for i in range(10):
    fila = df_nt.iloc[i]
    acts = fila['actions']
    n_acts = len(acts) if hasattr(acts, '__len__') else 0
    print(f"  Fila {i:02d} | Sala: {str(fila['venue']):<18} | Jugadores: {len(fila['players'])} | Total pasos: {n_acts}")

# 2. Filtrar las manos que sí provienen de salas online reales (Full Tilt y Absolute Poker)
manos_online = df_nt[df_nt['venue'].isin(['Full Tilt Poker', 'Absolute Poker'])].copy()
print(f"\n🎰 Total de manos de salas online reales disponibles: {len(manos_online):,}")

# 3. Tomar la primera mano online real
mano_real = manos_online.iloc[0]
print(f"\n🃏 MANO REAL SELECCIONADA:")
print(f"   Sala: {mano_real['venue']}")
print(f"   Jugadores: {mano_real['players']}")
print(f"   Fichas iniciales: {mano_real['starting_stacks']}")
print(f"   Ciegas: {mano_real['blinds_or_straddles']}")
print(f"   Total de acciones registradas: {len(mano_real['actions'])}")

# 4. Ver los primeros 20 pasos de esta mano
print("\n📜 SECUENCIA DE ACCIONES (Primeros 20 pasos):")
for j, accion in enumerate(mano_real['actions'][:20], 1):
    print(f"   Paso {j:02d}: {repr(accion)}")

🔍 LONGITUD DE 'actions' EN LAS PRIMERAS 10 FILAS:
  Fila 00 | Sala: nan                | Jugadores: 3 | Total pasos: 1
  Fila 01 | Sala: nan                | Jugadores: 6 | Total pasos: 16
  Fila 02 | Sala: nan                | Jugadores: 6 | Total pasos: 13
  Fila 03 | Sala: nan                | Jugadores: 6 | Total pasos: 24
  Fila 04 | Sala: nan                | Jugadores: 6 | Total pasos: 19
  Fila 05 | Sala: nan                | Jugadores: 6 | Total pasos: 23
  Fila 06 | Sala: nan                | Jugadores: 6 | Total pasos: 24
  Fila 07 | Sala: nan                | Jugadores: 6 | Total pasos: 13
  Fila 08 | Sala: nan                | Jugadores: 6 | Total pasos: 12
  Fila 09 | Sala: nan                | Jugadores: 6 | Total pasos: 24

🎰 Total de manos de salas online reales disponibles: 440,249

🃏 MANO REAL SELECCIONADA:
   Sala: Absolute Poker
   Jugadores: ['vwPa9NkyrlxHu1v4rBybHw' 'H7TvAHMQX+bVR81Nq796DQ'
 '+38WdZ4L8J7gkKBgk8Y+aQ' 'EW2DjH0/LLkb08fGS16R8g'
 '3wT3m+GDGtVWU1KR2MWJ

In [10]:
# 1. Ver la película completa de la mano real (los 15 pasos)
print("📜 HISTORIA COMPLETA DE LA PARTIDA:")
for num, act in enumerate(mano_real['actions'], 1):
    print(f"  Paso {num:02d}: {act}")

# 2. Auditar la presencia de cartas ocultas ('????') en una muestra de 1.000 manos
print("\n🔍 AUDITORÍA DE CARTAS PRIVADAS EN 1.000 MANOS (Doc. Maestro § 2.6):")
muestra_auditoria = 1000
con_cartas_ocultas = 0

for i in range(muestra_auditoria):
    acts = manos_online.iloc[i]['actions']
    texto_acciones = " ".join(acts)
    if "????" in texto_acciones:
        con_cartas_ocultas += 1

porcentaje_ocultas = (con_cartas_ocultas / muestra_auditoria) * 100
print(f"• Manos analizadas: {muestra_auditoria:,}")
print(f"• Manos con cartas privadas ocultas (????): {con_cartas_ocultas:,} ({porcentaje_ocultas:.1f}%)")

📜 HISTORIA COMPLETA DE LA PARTIDA:
  Paso 01: d dh p1 ????
  Paso 02: d dh p2 ????
  Paso 03: d dh p3 ????
  Paso 04: d dh p4 ????
  Paso 05: d dh p5 ????
  Paso 06: d dh p6 ????
  Paso 07: p3 f
  Paso 08: p4 cbr 40
  Paso 09: p5 f
  Paso 10: p6 cc
  Paso 11: p1 f
  Paso 12: p2 f
  Paso 13: d db Qd9c5c
  Paso 14: p4 cbr 88
  Paso 15: p6 f

🔍 AUDITORÍA DE CARTAS PRIVADAS EN 1.000 MANOS (Doc. Maestro § 2.6):
• Manos analizadas: 1,000
• Manos con cartas privadas ocultas (????): 1,000 (100.0%)


In [11]:
# Conclusiones finales y reporte de la auditoría
resumen_auditoria = {
    "Total manos descargadas (1 shard)": f"{len(df):,}",
    "Manos No-Limit Texas Hold'em (NT)": f"{len(df_nt):,} ({len(df_nt)/len(df)*100:.2f}%)",
    "Manos de salas online activas (Full Tilt / Absolute)": f"{len(manos_online):,}",
    "Disponibilidad de cartas privadas": "100% ocultas (????) durante el juego",
    "Total estimado de decisiones humanas disponibles": f"~{len(manos_online) * 4:,} decisiones",
    "Acciones principales observadas": "f (fold), cc (check/call), cbr (bet/raise)"
}

print("=" * 65)
print("  REPORTE OFICIAL DE AUDITORÍA (ENTREGABLE SEMANA 1)")
print("=" * 65)
for clave, valor in resumen_auditoria.items():
    print(f" • {clave:<45} : {valor}")
print("=" * 65)
print("\n✅ DECISIÓN METODOLÓGICA CONVALIDADA (Doc. Maestro § 2.6):")
print("  1. El Modelo Neuronal se entrenará con información pública de la mesa.")
print("  2. El Motor Matemático independiente calculará la equity de cartas privadas.")
print("  3. Se concluye con éxito el Notebook 01: Datos auditados y verificados.")

  REPORTE OFICIAL DE AUDITORÍA (ENTREGABLE SEMANA 1)
 • Total manos descargadas (1 shard)             : 450,337
 • Manos No-Limit Texas Hold'em (NT)             : 450,261 (99.98%)
 • Manos de salas online activas (Full Tilt / Absolute) : 440,249
 • Disponibilidad de cartas privadas             : 100% ocultas (????) durante el juego
 • Total estimado de decisiones humanas disponibles : ~1,760,996 decisiones
 • Acciones principales observadas               : f (fold), cc (check/call), cbr (bet/raise)

✅ DECISIÓN METODOLÓGICA CONVALIDADA (Doc. Maestro § 2.6):
  1. El Modelo Neuronal se entrenará con información pública de la mesa.
  2. El Motor Matemático independiente calculará la equity de cartas privadas.
  3. Se concluye con éxito el Notebook 01: Datos auditados y verificados.
